**Title**: Download File from Analysis Zip Archieve <br>
**Date**:  10/22/2020  
**Description**:  
To illustrate how to read DICOM Zip member directly in memory.


# Requirements
- TODO #####

Dataset from here https://openneuro.org/datasets/ds003047/versions/1.0.0

TODO - Show how to import the dataset to flywheel (maybe?)


1. Show how to install on Mac, Linux, Windows
- Give a direct link to the AWS CLI installer

2. Then download the dataset via this command `aws s3 sync --no-sign-request s3://openneuro.org/ds003216 ds003216-download/` - might take awhile to download this

3. Show the command line on how to import it to Flywheel 

# Install and Import Dependencies

In [ ]:
# Install specific packages required for this notebook
!pip install flywheel-sdk 

In [1]:
# Import packages
from getpass import getpass
import logging
import os
from pathlib import Path
import time

import flywheel
from permission import check_user_permission


In [2]:
# Instantiate a logger
logging.basicConfig(level=logging.INFO, format='%(asctime)s %(levelname)s %(message)s')
log = logging.getLogger('root')

# Flywheel API Key and Client

Get your API_KEY. More on this at in the Flywheel SDK doc [here](https://flywheel-io.gitlab.io/product/backend/sdk/branches/master/python/getting_started.html#api-key).

In [3]:
API_KEY = getpass('Enter API_KEY here: ')

Enter API_KEY here:  ····································


Instantiate the Flywheel API client

In [4]:
fw = flywheel.Client(API_KEY if 'API_KEY' in locals() else os.environ.get('FW_KEY'))

Show Flywheel logging information

In [5]:
log.info('You are now logged in as %s to %s', fw.get_current_user()['email'], fw.get_config()['site']['api_url'])

2020-10-28 10:51:36,502 INFO You are now logged in as huiqiantan@flywheel.io to https://ss.ce.flywheel.io:443/api


***

# Initialize a few values

Define your Project's Label and let's look for it on your Flywheel instance.

In [ ]:
PROJECT_LABEL = input('Please enter your Project Label here: ')

In [ ]:
project = fw.projects.find_first(f'label={PROJECT_LABEL}')

***

# Requirements

Before starting off, we want to check your permission on the Flywheel Instance in order to proceed in this notebook. 

In [61]:
min_reqs = {
"site": "user",
"group": "ro",
"project": ['analyses_view_metadata','files_view_metadata','files_view_contents','files_download']
}

<div class="alert alert-block alert-info" style="color:black"><b>Tip:</b> Group ID and Project Label can be found on top of the Project page on the Flywheel Instance as shown in the snippet below.</div>


In [ ]:
GROUP_ID = input('Please enter the Group ID that you will be working with: ')

`check_user_permission` will return True if both the group and project meet the minimum requirement, else a compatible list will be printed.

In [ ]:
check_user_permission(fw, min_reqs, group=GROUP_ID, project=PROJECT_LABEL)

***

# Get analyses container

First, we get the Subject container first. In this tutorial we will be working with `sub-04`.

In [62]:
subj = project.subjects.find_first('label=sub-04')

Loop through each session to find which session container contains analysis

In [66]:
for sess in subj.sessions.iter():
    print(sess['label'])

ses-1


It seems like there is only one Session container for Subject `sub-04`.
Now, we will loop through the files within the analyses container in the Session container

In [69]:
session_container = subj.sessions()[0]

In [51]:
analysis = session_container.reload()['analyses'][0]

In [73]:
for file in analysis['files']:
    print(file.name)

AFQ-Browser.html.zip
AFQ_Output_afq_24-Oct-2020_01h40m12s.zip
AFQ_ad.csv
AFQ_cl.csv
AFQ_curvature.csv
AFQ_fa.csv
AFQ_md.csv
AFQ_params.json
AFQ_rd.csv
AFQ_torsion.csv
AFQ_volume.csv
Reproduce.json
Reproduce.mat
afq_24-Oct-2020_01h40m12s.mat
dtiInit.json
dtiInit.log
dtiInit_24-Oct-2020_01-23-37.zip


We use `reload()` method here to retrieve a complete view of the data within the Session container. We are going to work with the second zip file.

In [74]:
zip_file_name = analysis['files'][1].name

In [78]:
analysis_id = analysis.id
analysis_id

'5f938175a15996c47935f5ef'

We will be using the `get_file_zip_info` method to retrieve the files information within the zip file.

In [79]:
zip_info = analysis.get_file_zip_info(zip_file_name)

In [82]:
len(zip_info.members)

92

There is a total of 92 files in the zip files.

In [53]:
zip_file_name = session_container.reload()['analyses'][0]['files'][1].name

In [50]:
analysis_id = session_container.reload()['analyses'][0].id
analysis_id

'5f938175a15996c47935f5ef'

In [55]:
zip_info = analysis.get_file_zip_info(zip_file_name)

In [56]:
len(zip_info.members)

92

Let's see how many directory there is and how many nifti file output

In [105]:
for ii, file in enumerate(zip_info.members):
    if file.path[-1] != '/':
        if file.path.endswith('nii.gz'):
            print(f'{ii} {Path(file.path).name}')
        
    else:
        print(f'dir {file.path}'      )

dir afq_24-Oct-2020_01h40m12s/
dir afq_24-Oct-2020_01h40m12s/csv_files/
dir afq_24-Oct-2020_01h40m12s/dti64trilin/
dir afq_24-Oct-2020_01h40m12s/dti64trilin/mrtrix/
dir afq_24-Oct-2020_01h40m12s/dti64trilin/fibers/
dir afq_24-Oct-2020_01h40m12s/dti64trilin/fibers/conTrack/
dir afq_24-Oct-2020_01h40m12s/dti64trilin/ROIs/
dir afq_24-Oct-2020_01h40m12s/dti64trilin/LiFE/
dir afq_24-Oct-2020_01h40m12s/dti64trilin/bin/
82 brainMask.nii.gz
83 wmProb.nii.gz
84 tensors.nii.gz
85 mdStd.nii.gz
86 vectorRGB.nii.gz
87 pddDispersion.nii.gz
88 b0.nii.gz
89 faStd.nii.gz
90 wmMask.nii.gz


Now, we will be downloading the file's zip member by using the `download_file_zip_member` method. We will be chosing the 86th member in the zip file to download.

In [60]:
analysis.download_file_zip_member?

Signature: analysis.download_file_zip_member(file_name, member_path, dest_file)
Docstring: Download file's zip member to the given path
File:      ~/opt/anaconda3/lib/python3.7/site-packages/flywheel/models/mixins.py
Type:      method


In [108]:
filename = 'test01'
member_path = zip_info.members[86].path
dest_file = Path(zip_info.members[86].path).name
print(f'{filename}\n{member_path}\n{dest_file}')
analysis.download_file_zip_member(zip_file_name, member_path, dest_file)

test01
afq_24-Oct-2020_01h40m12s/dti64trilin/bin/vectorRGB.nii.gz
vectorRGB.nii.gz
